# Demo 04 : Command and Send

Run the cells in order and watch. There is nothing to write. No model runs here ; section b plays the real game.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## a · command

A bug comes in. `triage` decides who fixes it and returns the state update and the next node together.

In [2]:
from typing import Literal, TypedDict
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command

class Bug(TypedDict):
    kind: str
    assigned: str

def triage(state: Bug) -> Command[Literal["coder", "artist"]]:
    target = "artist" if state["kind"] == "sprite" else "coder"
    return Command(update={"assigned": target}, goto=target)

builder = StateGraph(Bug)
builder.add_node("triage", triage)
builder.add_node("coder", lambda s: {})
builder.add_node("artist", lambda s: {})
builder.add_edge(START, "triage")
builder.add_edge("coder", END)
builder.add_edge("artist", END)
print("route:", builder.compile().invoke({"kind": "sprite", "assigned": ""})["assigned"])

route: artist


## b · send-map-reduce

The bot from the game kit plays a finished Comet Snake. A level here is one start seed : `init(seed)` lays out a
different game. We first load the game and write one function that plays one level.

In [3]:
import json

import importlib.util
from pathlib import Path

kit = Path(data_folder("game-studio/kit"))
spec = importlib.util.spec_from_file_location("harness", kit / "harness.py")
harness = importlib.util.module_from_spec(spec)
spec.loader.exec_module(harness)

TEMPLATE = (kit / "template.html").read_text(encoding="utf-8")
RULES = (kit / "rules.md").read_text(encoding="utf-8")
TWISTS = json.loads((kit / "twists.json").read_text(encoding="utf-8"))
BRIEFS = json.loads((kit / "briefs.json").read_text(encoding="utf-8"))

In [4]:
saved = json.loads(Path(data("game-studio/saved-run.json")).read_text(encoding="utf-8"))
game_file = harness.splice(TEMPLATE, saved["board"]["logic"])

def play(level: int) -> dict:
    with harness.Runtime(memory_limit=64 * 1024 * 1024) as runtime, runtime.new_context(timeout=20.0) as ctx:
        return harness.Game(ctx, game_file).play(level)

print(play(1))

{'eaten': 56, 'score': 56, 'ticks': 895, 'reason': 'self', 'tickMs': 60, 'length': 59, 'comets': 6}


The list of levels is on the board, so the number of testers is known only when the graph runs. `fan_out`
creates one `Send` per level, each with its own input. The reducer adds every report to one list, and `collect` sums up.

In [5]:
import operator
from typing import Annotated, TypedDict
from langgraph.graph import END, START, StateGraph
from langgraph.types import Send

class Summary(TypedDict):
    levels: list[int]
    reports: Annotated[list[str], operator.add]
    summary: str

class Level(TypedDict):
    level: int

def fan_out(state: Summary):
    return [Send("tester", {"level": level}) for level in state["levels"]]

def tester(state: Level) -> dict:
    game = play(state["level"])
    return {"reports": [f"level {state['level']}: {game['eaten']} stars"]}

def collect(state: Summary) -> dict:
    return {"summary": f"{len(state['reports'])} levels tested"}

builder = StateGraph(Summary)
builder.add_node("tester", tester)
builder.add_node("collect", collect)
builder.add_conditional_edges(START, fan_out, ["tester"])
builder.add_edge("tester", "collect")
builder.add_edge("collect", END)
graph = builder.compile()

for levels in ([1, 2, 3], [1, 2, 3, 4, 5]):
    result = graph.invoke({"levels": levels, "reports": [], "summary": ""})
    print("reports:", result["reports"])
    print("summary:", result["summary"])

reports: ['level 1: 56 stars', 'level 2: 66 stars', 'level 3: 65 stars']
summary: 3 levels tested


reports: ['level 1: 56 stars', 'level 2: 66 stars', 'level 3: 65 stars', 'level 4: 80 stars', 'level 5: 63 stars']
summary: 5 levels tested


## c · deferred-nodes

Two branches of different length join at `splice`, which puts sprites and code in one file. The `artist` branch is one
step, `coder` then `reviewer` are two. With plain edges `splice` runs as soon as any branch reaches it. With
`defer=True` it waits until no other work is left.

In [6]:
class Log(TypedDict):
    log: Annotated[list[str], operator.add]

def uneven(defer: bool):
    g = StateGraph(Log)
    g.add_node("artist", lambda s: {"log": ["artist"]})
    g.add_node("coder", lambda s: {"log": ["coder"]})
    g.add_node("reviewer", lambda s: {"log": ["reviewer"]})
    g.add_node("splice", lambda s: {"log": [f"splice saw {len(s['log'])}"]}, defer=defer)
    g.add_edge(START, "artist")
    g.add_edge(START, "coder")
    g.add_edge("coder", "reviewer")
    g.add_edge("artist", "splice")
    g.add_edge("reviewer", "splice")
    g.add_edge("splice", END)
    return g.compile()

print("plain   :", uneven(False).invoke({"log": []})["log"])
print("deferred:", uneven(True).invoke({"log": []})["log"])

plain   : ['artist', 'coder', 'reviewer', 'splice saw 2', 'splice saw 4']
deferred: ['artist', 'coder', 'reviewer', 'splice saw 3']


The plain `splice` runs twice and the first run sees only two entries. The deferred `splice` runs once, after both
branches. The same flag works on a `Send` fan-out :

In [7]:
deferred = StateGraph(Summary)
deferred.add_node("tester", tester)
deferred.add_node("collect", collect, defer=True)
deferred.add_conditional_edges(START, fan_out, ["tester"])
deferred.add_edge("tester", "collect")
deferred.add_edge("collect", END)
result = deferred.compile().invoke({"levels": [4, 5], "reports": [], "summary": ""})
print("deferred summary:", result["summary"])

deferred summary: 2 levels tested


## d · supersteps

A superstep is one round of nodes that run at the same time. We build the full crew and print each round.
Three Designers share a round, and so do the two Testers. Each node sleeps for 0.3 seconds and logs how many entries
it saw. A join built from `add_edge([...], "gate")` waits for both Testers.

In [8]:
import time

def work(name: str):
    def run(state: Log) -> dict:
        time.sleep(0.3)
        return {"log": [f"{name} saw {len(state['log'])}"]}
    return run

studio = StateGraph(Log)
for name in ("designer_a", "designer_b", "designer_c", "producer", "coder", "rule_tester", "bot_tester", "gate"):
    studio.add_node(name, work(name))
studio.add_edge(START, "designer_a")
studio.add_edge(START, "designer_b")
studio.add_edge(START, "designer_c")
studio.add_edge(["designer_a", "designer_b", "designer_c"], "producer")
studio.add_edge("producer", "coder")
studio.add_edge("coder", "rule_tester")
studio.add_edge("coder", "bot_tester")
studio.add_edge(["rule_tester", "bot_tester"], "gate")
studio.add_edge("gate", END)
line = studio.compile()

start = time.perf_counter()
rounds = {}
for event in line.stream({"log": []}, stream_mode="debug"):
    if event["type"] == "task_result":
        rounds.setdefault(event["step"], []).extend(dict(event["payload"]["result"])["log"])
for number, entries in rounds.items():
    print(f"superstep {number}:", sorted(entries))
print(f"8 nodes of 0.3 s in {time.perf_counter() - start:.1f} s")

superstep 1: ['designer_a saw 0', 'designer_b saw 0', 'designer_c saw 0']
superstep 2: ['producer saw 3']
superstep 3: ['coder saw 4']
superstep 4: ['bot_tester saw 5', 'rule_tester saw 5']
superstep 5: ['gate saw 7']
8 nodes of 0.3 s in 1.5 s


## Things to try

- Add a sixth level to the map-reduce input
- Change the collected report format and run both graphs again
- Give one Designer a sleep of 1 second and read the time